In [ ]:
#------------------------------------------------ Import Lib ----------------------------------------
import re
import os
import datetime
from time import sleep

import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [ ]:
#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'LS CBLE' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

try:
    scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## production environment
except NameError:
    scriptfolder=os.getcwd() ## notebook environment

os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


In [ ]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': []}


regdict={

        regulatorName+' 1': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/commercial-banks-and-forex-agencies/',
        regulatorName+' 2': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/insurance-brokers/',
        regulatorName+' 3': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/micro-finance-institutions/',
        regulatorName+' 4': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/insurance-companies/',
        regulatorName+' 5': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/asset-managers-in-lesotho/',
        regulatorName+' 6': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/credit-bureau/',
        regulatorName+' 7': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/foreign-exchange-bureau-and-money-transfere/',
        regulatorName+' 8': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/pension-fund-administrators/',
        regulatorName+' 9': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/pension-fund/',
        regulatorName+' 10': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/mobile-money-issuers-in-lesotho/',
        regulatorName+' 11': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/pension-fund-intermediaries/',
        regulatorName+' 12': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/stock-brokers/',
        regulatorName+' 13': 'https://centralbank.org.ls/other-financial-institutions-2/wpbdp_category/trustees/',
        }


Typology={

       regulatorName + ' 1': 'Commercial Banks and Forex Agencies',
       regulatorName + ' 2': 'Insurance Brokers',
       regulatorName + ' 3': 'Micro Finance Institutions',
       regulatorName + ' 4': 'Insurance Companies',
       regulatorName + ' 5': 'Asset Managers in Lesotho',
       regulatorName + ' 6': 'Credit Bureaus',
       regulatorName + ' 7': 'Foreign Exchange Bureau and Money Transfer',
       regulatorName + ' 8': 'Pension Fund Administrators',
       regulatorName + ' 9': 'Pension Funds',
       regulatorName + ' 10': 'Mobile Money Issuers',
       regulatorName + ' 11': 'Pension Fund Intermediaries',
       regulatorName + ' 12': 'Stock Brokers',
       regulatorName + ' 13': 'Trustees',

        }


# ListLabel rule (per ticket owner): 1 = bank named in list name, 2 = insurance,
# 3 = bank & insurance, 4 = other
ListLabeldict={

       regulatorName + ' 1': 1,   # Commercial Banks and Forex Agencies
       regulatorName + ' 2': 2,   # Insurance Brokers
       regulatorName + ' 3': 4,   # Micro Finance Institutions
       regulatorName + ' 4': 2,   # Insurance Companies
       regulatorName + ' 5': 4,   # Asset Managers in Lesotho
       regulatorName + ' 6': 4,   # Credit Bureaus
       regulatorName + ' 7': 4,   # Foreign Exchange Bureau and Money Transfer
       regulatorName + ' 8': 4,   # Pension Fund Administrators
       regulatorName + ' 9': 4,   # Pension Funds
       regulatorName + ' 10': 4,  # Mobile Money Issuers
       regulatorName + ' 11': 4,  # Pension Fund Intermediaries
       regulatorName + ' 12': 4,  # Stock Brokers
       regulatorName + ' 13': 4,  # Trustees

        }


processdate = now.strftime('%Y-%m-%d')

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
})

In [ ]:
#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


def clean_text(text):

    s = (text or '').replace('\xa0', ' ')

    s = re.sub(r"\s+", " ", s).strip()

    return s


In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------

# Each regdict URL is a WordPress Business Directory (wpbdp) category page.
# Listings are cards (div.wpbdp-listing) with title / CEO / phone / address on the card.
# Pages paginate 10 listings per page: follow the '.wpbdp-pagination .next a' link until it disappears.

for k, reg in enumerate(regdict):
    rows_before = len(sqldict['Name'])
    url = regdict[reg]
    seen_listing_ids = set()  # guard against pagination overlap
    pages = 0
    while url and pages < 60:
        resp = session.get(url, verify=False, timeout=60)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'lxml')
        pages += 1
        for card in soup.select('.wpbdp-listing'):
            listing_id = card.get('id', '')
            if listing_id and listing_id in seen_listing_ids:
                continue
            if listing_id:
                seen_listing_ids.add(listing_id)

            title_el = card.select_one('.listing-title h3')
            name_ = clean_text(title_el.get_text(' ', strip=True)) if title_el else ''
            if not name_:
                continue

            phone_el = card.select_one('.wpbdp-field-phone .value')
            phone_ = clean_text(phone_el.get_text(' ', strip=True)) if phone_el else ''

            addr_el = card.select_one('.address-info div')
            address_ = clean_text(addr_el.get_text(' ', strip=True)) if addr_el else ''

            sqldict['Name'].append(name_)
            sqldict['Address_1'].append(address_)
            sqldict['Phone'].append(phone_)
            sqldict['ListLabel'].append(ListLabeldict[reg])
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['RegulationType'].append('Regulated')

        next_el = soup.select_one('.wpbdp-pagination .next a')
        url = urljoin(url, next_el['href']) if next_el else None
        sleep(1)

    sqldict = bourange_same_length_array(sqldict)
    rows_after = len(sqldict['Name'])
    print(f"[INFO] : {reg} ({Typology[reg]}) pages={pages} rows={rows_after - rows_before}")

In [ ]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)

df = df[df['Name']!='']

# The Insurance Brokers category contains a bare test listing published by the site
# admin ('Test New': no CEO, phone or address). It is not a real entity: drop it.
df = df[df['Name'].str.strip().str.lower() != 'test new']

# The site itself contains a few duplicate listing posts (same entity published twice,
# e.g. 'boikaho-financial-services' and 'boikaho-financial-services-2'): keep the first.
df = df.drop_duplicates(subset=['ListCode', 'Name', 'Address_1'], keep='first')

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

# print(f"[INFO] : saved {len(df)} rows to {os.path.join(scriptfolder, filename)}")
